# 02 - Tiền xử lý ảnh + Ablation Study

**Mục tiêu:** Áp dụng 4 nhóm kỹ thuật tiền xử lý, mỗi nhóm kèm ablation study với metric định lượng.

**Dataset:** NWPU-RESISC45 (45 lớp, 256x256 pixel)

## 0. Setup

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
from scipy import stats
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 11, 'figure.dpi': 100})
sns.set_style("whitegrid")

TRAIN_DIR = r"D:\DataMining\Dataset\train"
classes = sorted(os.listdir(TRAIN_DIR))
print(f"Loaded: {len(classes)} lop")

In [ ]:
def load_sample(n_per_class=20, target_classes=None, seed=42):
    """Load sample ảnh, trả về list (img_rgb, class_name)."""
    np.random.seed(seed)
    target = target_classes or classes
    samples = []
    for cls in target:
        paths = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))
        chosen = np.random.choice(paths, min(n_per_class, len(paths)), replace=False)
        for p in chosen:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            samples.append((img, cls))
    return samples

### Chọn lớp đại diện cho ablation study

Quét nhanh mean brightness 45 lớp (5 ảnh/lớp), sắp xếp tăng dần,
chọn 10 lớp cách đều để đảm bảo phủ đa dạng brightness.

In [ ]:
print("Quet nhanh brightness per class (5 anh/lop)..")
quick_brightness = {}
for cls in classes:
    paths = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))[:5]
    vals = [cv2.imread(p).mean() for p in paths]
    quick_brightness[cls] = np.mean(vals)

sorted_by_bright = sorted(quick_brightness, key=quick_brightness.get)
n = len(sorted_by_bright)
indices_10 = np.linspace(0, n - 1, 10, dtype=int)
SAMPLE_CLASSES = [sorted_by_bright[i] for i in indices_10]

print("10 lop dai dien (brightness thap -> cao):")
for cls in SAMPLE_CLASSES:
    print(f"  {cls}: brightness = {quick_brightness[cls]:.1f}")

---
## 1. Resize - Ablation Study (SSIM / PSNR)

**H0:** Chat luong anh (SSIM) khong khac biet giua 3 kich thuoc resize.

**H1:** It nhat 1 kich thuoc cho SSIM khac biet.

In [ ]:
RESIZE_DIMS = [(64, 64), (128, 128), (224, 224)]

samples = load_sample(n_per_class=30, target_classes=SAMPLE_CLASSES)
print(f"Sample: {len(samples)} anh tu {len(SAMPLE_CLASSES)} lop")

In [ ]:
results_resize = []

for img_orig, cls in tqdm(samples, desc="Resize ablation"):
    for (h, w) in RESIZE_DIMS:
        resized = cv2.resize(img_orig, (w, h), interpolation=cv2.INTER_AREA)
        restored = cv2.resize(resized, (256, 256), interpolation=cv2.INTER_LINEAR)

        s = ssim(img_orig, restored, channel_axis=2)
        p = psnr(img_orig, restored)

        results_resize.append({
            'class': cls, 'size': f"{h}x{w}",
            'ssim': s, 'psnr': p
        })

df_resize = pd.DataFrame(results_resize)

summary = df_resize.groupby('size').agg(
    ssim_mean=('ssim', 'mean'), ssim_std=('ssim', 'std'),
    psnr_mean=('psnr', 'mean'), psnr_std=('psnr', 'std')
).round(4)
print("Ablation Resize:")
print(summary.to_string())

In [ ]:
# Kiểm định ANOVA + post-hoc trên SSIM
groups_ssim = {s: df_resize[df_resize['size']==s]['ssim'].values for s in ['64x64', '128x128', '224x224']}

# Levene test
lev_s, lev_p = stats.levene(*groups_ssim.values())
print(f"Levene test: stat={lev_s:.2f}, p={lev_p:.2e} -> Variance {'KHONG dong nhat' if lev_p < 0.05 else 'dong nhat'}")

# ANOVA
f_val, p_val = stats.f_oneway(*groups_ssim.values())
print(f"ANOVA (SSIM ~ Resize): F={f_val:.2f}, p={p_val:.2e} -> {'BAC BO H0' if p_val < 0.05 else 'KHONG BAC BO H0'}")

# Eta-squared
all_ssim = np.concatenate(list(groups_ssim.values()))
gm = all_ssim.mean()
ss_b = sum(len(g) * (g.mean() - gm)**2 for g in groups_ssim.values())
ss_t = np.sum((all_ssim - gm)**2)
eta2_resize = ss_b / ss_t
print(f"Eta² = {eta2_resize:.3f} (effect size: {'LON' if eta2_resize>0.14 else 'TRUNG BINH' if eta2_resize>0.06 else 'NHO'})")

# Post-hoc: pairwise Mann-Whitney (3 cặp, Bonferroni)
print(f"\nPost-hoc (Mann-Whitney, Bonferroni alpha={0.05/3:.4f}):")
sizes = ['64x64', '128x128', '224x224']
for i in range(len(sizes)):
    for j in range(i+1, len(sizes)):
        u, p = stats.mannwhitneyu(groups_ssim[sizes[i]], groups_ssim[sizes[j]], alternative='two-sided')
        sig = "KHAC BIET" if p < 0.05/3 else "Khong khac biet"
        print(f"  {sizes[i]} vs {sizes[j]}: U={u:.0f}, p={p:.2e} -> {sig}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df_resize, x='size', y='ssim', ax=axes[0], palette='Blues_d',
            order=['64x64', '128x128', '224x224'])
axes[0].set_title("SSIM theo kich thuoc resize")
axes[0].set_ylabel("SSIM")

sns.boxplot(data=df_resize, x='size', y='psnr', ax=axes[1], palette='Oranges_d',
            order=['64x64', '128x128', '224x224'])
axes[1].set_title("PSNR theo kich thuoc resize (dB)")
axes[1].set_ylabel("PSNR (dB)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, (img, cls) in enumerate([(samples[0][0], samples[0][1]), (samples[60][0], samples[60][1])]):
    axes[row][0].imshow(img)
    axes[row][0].set_title(f"Goc 256x256\n({cls})", fontsize=9)
    axes[row][0].axis('off')
    for col, (h, w) in enumerate(RESIZE_DIMS):
        resized = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
        axes[row][col+1].imshow(resized)
        s = ssim(img, cv2.resize(resized, (256,256), interpolation=cv2.INTER_LINEAR), channel_axis=2)
        axes[row][col+1].set_title(f"{h}x{w}\nSSIM={s:.3f}", fontsize=9)
        axes[row][col+1].axis('off')
plt.suptitle("So sanh chat luong anh qua cac kich thuoc resize", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\n=> KET LUAN: 224x224 giu chat luong tot nhat (SSIM cao nhat)")
print(f"   Moi cap kich thuoc deu khac biet co y nghia (post-hoc p < {0.05/3:.4f})")
print(f"   Phu hop lam input cho pretrained CNNs (ResNet, VGG)")

---
## 2. Color Space - Ablation Study (PCA Explained Variance)

So sánh RGB, HSV, Lab: color space nào nén thông tin tốt nhất?

In [ ]:
COLOR_SPACES = {
    'RGB': lambda img: img,
    'HSV': lambda img: cv2.cvtColor(img, cv2.COLOR_RGB2HSV),
    'Lab': lambda img: cv2.cvtColor(img, cv2.COLOR_RGB2Lab)
}

pca_results = {}
n_pca_sample = 500
pca_samples = load_sample(n_per_class=12, target_classes=SAMPLE_CLASSES)[:n_pca_sample]

for cs_name, convert_fn in COLOR_SPACES.items():
    print(f"PCA tren {cs_name}..")
    features = []
    for img, _ in pca_samples:
        converted = convert_fn(img)
        features.append(converted.reshape(-1).astype(np.float32))

    X = np.array(features)
    X = StandardScaler().fit_transform(X)

    pca = PCA(n_components=min(50, X.shape[0]-1))
    pca.fit(X)
    pca_results[cs_name] = pca.explained_variance_ratio_

print("Done")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cs_name, evr in pca_results.items():
    axes[0].plot(range(1, len(evr)+1), evr[:50], marker='.', label=cs_name, markersize=4)
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Scree Plot")
axes[0].legend()
axes[0].set_xlim(0, 30)

for cs_name, evr in pca_results.items():
    axes[1].plot(range(1, len(evr)+1), np.cumsum(evr[:50]), marker='.', label=cs_name, markersize=4)
axes[1].axhline(0.95, color='red', linestyle='--', alpha=0.5, label='95% threshold')
axes[1].set_xlabel("So components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Explained Variance")
axes[1].legend()
axes[1].set_xlim(0, 30)

plt.tight_layout()
plt.show()

In [ ]:
print("So components can de giu X% variance:")
print(f"{'Color Space':<12} {'90%':>8} {'95%':>8} {'99%':>8}")
for cs_name, evr in pca_results.items():
    cum = np.cumsum(evr)
    n90 = np.argmax(cum >= 0.90) + 1
    n95 = np.argmax(cum >= 0.95) + 1
    n99 = np.argmax(cum >= 0.99) + 1 if cum[-1] >= 0.99 else ">50"
    print(f"{cs_name:<12} {n90:>8} {n95:>8} {str(n99):>8}")

---
## 3. Normalization - Ablation Study (KS test)

**H0:** Phan bo pixel sau normalize khong khac voi phan bo chuan N(0,1).

**H1:** Phan bo pixel sau normalize khac voi N(0,1).

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

def norm_minmax(img):
    return img.astype(np.float32) / 255.0

def norm_zscore(img):
    img_f = img.astype(np.float32)
    return (img_f - img_f.mean()) / (img_f.std() + 1e-8)

def norm_imagenet(img):
    img_f = img.astype(np.float32) / 255.0
    return (img_f - IMAGENET_MEAN) / IMAGENET_STD

def norm_histeq(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2Lab)
    lab[:,:,0] = cv2.equalizeHist(lab[:,:,0])
    return cv2.cvtColor(lab, cv2.COLOR_Lab2RGB).astype(np.float32) / 255.0

NORM_METHODS = {
    'Original': lambda x: x.astype(np.float32) / 255.0,
    'Min-Max [0,1]': norm_minmax,
    'Z-Score': norm_zscore,
    'ImageNet': norm_imagenet,
    'HistEq': norm_histeq
}

In [ ]:
norm_samples = load_sample(n_per_class=10, target_classes=SAMPLE_CLASSES[:5])
print(f"Normalization test: {len(norm_samples)} anh")

norm_pixels = {}
for method_name, norm_fn in NORM_METHODS.items():
    all_px = []
    for img, _ in norm_samples:
        normed = norm_fn(img)
        all_px.append(normed.ravel())
    norm_pixels[method_name] = np.concatenate(all_px)

In [ ]:
# KS test + Levene test giua cac phuong phap
print("KS Test (vs Normal distribution N(0,1)):")
print(f"{'Method':<16} {'KS stat':>10} {'p-value':>12} {'Mean':>8} {'Std':>8} {'-> Normal?':>12}")
print("-" * 70)

for method, px in norm_pixels.items():
    px_sample = np.random.choice(px, min(100000, len(px)), replace=False)
    px_std = (px_sample - px_sample.mean()) / (px_sample.std() + 1e-8)
    ks_stat, ks_p = stats.kstest(px_std, 'norm')
    is_normal = "Gan chuan" if ks_stat < 0.05 else "Khong chuan"
    print(f"{method:<16} {ks_stat:>10.4f} {ks_p:>12.2e} {px_sample.mean():>8.4f} {px_sample.std():>8.4f} {is_normal:>12}")

print("\nNote: Voi sample lon (100k), KS test rat nhay -> p-value thuong rat nho.")
print("      Dung KS statistic de so sanh: stat cang NHO -> cang gan normal.")

In [ ]:
# So sanh variance giua cac phuong phap (Levene test)
norm_groups = [np.random.choice(px, 10000, replace=False) for px in norm_pixels.values()]
lev_s, lev_p = stats.levene(*norm_groups)
print(f"Levene test (variance giua 5 phuong phap): stat={lev_s:.2f}, p={lev_p:.2e}")
print(f"  -> Variance {'KHAC NHAU' if lev_p < 0.05 else 'DONG NHAT'} giua cac phuong phap")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for idx, (method, px) in enumerate(norm_pixels.items()):
    row, col = divmod(idx, 3)
    px_plot = np.random.choice(px, 200000, replace=False) if len(px) > 200000 else px
    axes[row][col].hist(px_plot, bins=100, color='steelblue', alpha=0.7, density=True, edgecolor='none')
    axes[row][col].set_title(method, fontsize=11)
    axes[row][col].set_xlabel("Pixel value")
    axes[row][col].set_ylabel("Density")

if len(norm_pixels) < 6:
    axes[1][2].axis('off')
plt.suptitle("Phan bo pixel sau moi phuong phap Normalization", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
sample_img = norm_samples[0][0]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for idx, (method, norm_fn) in enumerate(NORM_METHODS.items()):
    normed = norm_fn(sample_img)
    if method in ('Z-Score', 'ImageNet'):
        display = np.clip((normed - normed.min()) / (normed.max() - normed.min() + 1e-8), 0, 1)
    else:
        display = np.clip(normed, 0, 1)
    axes[idx].imshow(display)
    axes[idx].set_title(method, fontsize=9)
    axes[idx].axis('off')
plt.suptitle("Anh mau sau moi phuong phap Normalization", fontsize=13)
plt.tight_layout()
plt.show()

print("\n=> KET LUAN: Z-Score va ImageNet co KS stat thap nhat -> gan normal nhat")
print("   ImageNet phu hop cho pretrained models (ResNet, VGG) vi dung cung scale")

---
## 4. Data Augmentation - Ablation Study (t-SNE)

**H0:** Augmentation khong lam thay doi intra-class variance (diversity).

**H1:** Augmentation lam tang intra-class variance.

In [ ]:
def augment_hflip(img):
    return cv2.flip(img, 1)

def augment_rotate(img, angle=15):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h))

def augment_jitter(img, brightness=30, contrast=0.2):
    img_f = img.astype(np.float32)
    img_f = img_f + np.random.uniform(-brightness, brightness)
    img_f = img_f * np.random.uniform(1-contrast, 1+contrast)
    return np.clip(img_f, 0, 255).astype(np.uint8)

def augment_crop(img, scale=0.8):
    h, w = img.shape[:2]
    new_h, new_w = int(h*scale), int(w*scale)
    top = np.random.randint(0, h - new_h + 1)
    left = np.random.randint(0, w - new_w + 1)
    cropped = img[top:top+new_h, left:left+new_w]
    return cv2.resize(cropped, (w, h))

def augment_blur(img, ksize=5):
    return cv2.GaussianBlur(img, (ksize, ksize), 0)

AUGMENTATIONS = {
    'Horizontal Flip': augment_hflip,
    'Rotation +/-15': augment_rotate,
    'Color Jitter': augment_jitter,
    'Random Crop': augment_crop,
    'Gaussian Blur': augment_blur
}

In [ ]:
vis_sample = load_sample(n_per_class=1, target_classes=SAMPLE_CLASSES[:3])

fig, axes = plt.subplots(3, 6, figsize=(16, 7))
for row, (img, cls) in enumerate(vis_sample[:3]):
    axes[row][0].imshow(img)
    axes[row][0].set_title(f"Original\n({cls})", fontsize=8)
    axes[row][0].axis('off')
    for col, (aug_name, aug_fn) in enumerate(AUGMENTATIONS.items()):
        np.random.seed(42)
        augmented = aug_fn(img)
        axes[row][col+1].imshow(augmented)
        axes[row][col+1].set_title(aug_name, fontsize=8)
        axes[row][col+1].axis('off')
plt.suptitle("5 phep Augmentation tren anh mau", fontsize=13)
plt.tight_layout()
plt.show()

### t-SNE: so sánh feature space trước/sau augmentation

Chọn 5 lớp cho t-SNE từ SAMPLE_CLASSES (vị trí cách đều theo brightness).

In [ ]:
tsne_pick = [0, 2, 5, 7, 9]
TSNE_CLASSES = [SAMPLE_CLASSES[i] for i in tsne_pick]
print(f"5 lop cho t-SNE: {TSNE_CLASSES}")

In [ ]:
print("Dang trich xuat features cho t-SNE..")
tsne_samples = load_sample(n_per_class=40, target_classes=TSNE_CLASSES)

def extract_feature(img):
    small = cv2.resize(img, (32, 32))
    return small.reshape(-1).astype(np.float32) / 255.0

orig_features, orig_labels = [], []
for img, cls in tsne_samples:
    orig_features.append(extract_feature(img))
    orig_labels.append(cls)

aug_features, aug_labels = [], []
np.random.seed(42)
aug_fns = list(AUGMENTATIONS.values())
for img, cls in tsne_samples:
    aug_img = img.copy()
    for fn in np.random.choice(aug_fns, 2, replace=False):
        aug_img = fn(aug_img)
    aug_features.append(extract_feature(aug_img))
    aug_labels.append(cls)

X_orig = np.array(orig_features)
X_aug = np.array(aug_features)
y_orig = np.array(orig_labels)
y_aug = np.array(aug_labels)

print(f"Original: {X_orig.shape}, Augmented: {X_aug.shape}")

In [ ]:
print("Running t-SNE (co the mat 1-2 phut)..")
X_combined = np.vstack([X_orig, X_aug])
is_augmented = np.array([0]*len(X_orig) + [1]*len(X_aug))
labels_combined = np.concatenate([y_orig, y_aug])

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_combined)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

mask_orig = is_augmented == 0
for cls in TSNE_CLASSES:
    cls_mask = (labels_combined == cls) & mask_orig
    axes[0].scatter(X_tsne[cls_mask, 0], X_tsne[cls_mask, 1], label=cls, alpha=0.6, s=20)
axes[0].set_title("Truoc Augmentation")
axes[0].legend(fontsize=8)

mask_aug = is_augmented == 1
for cls in TSNE_CLASSES:
    cls_mask = (labels_combined == cls) & mask_aug
    axes[1].scatter(X_tsne[cls_mask, 0], X_tsne[cls_mask, 1], label=cls, alpha=0.6, s=20)
axes[1].set_title("Sau Augmentation (pipeline)")
axes[1].legend(fontsize=8)

plt.suptitle("t-SNE: Feature Space truoc/sau Data Augmentation", fontsize=13)
plt.tight_layout()
plt.show()

### Kiểm định: Augmentation có tăng diversity?

In [ ]:
print("Intra-class variance truoc/sau augmentation:")
print(f"{'Class':<20} {'Original':>10} {'Augmented':>10} {'Change':>10}")
print("-" * 52)

orig_vars, aug_vars = [], []
for cls in TSNE_CLASSES:
    var_o = np.var(X_orig[y_orig == cls], axis=0).mean()
    var_a = np.var(X_aug[y_aug == cls], axis=0).mean()
    change = (var_a - var_o) / var_o * 100
    orig_vars.append(var_o)
    aug_vars.append(var_a)
    print(f"{cls:<20} {var_o:>10.4f} {var_a:>10.4f} {change:>+9.1f}%")

# Paired t-test
t_stat, t_p = stats.ttest_rel(orig_vars, aug_vars)

# Wilcoxon signed-rank (non-parametric alternative)
w_stat, w_p = stats.wilcoxon(orig_vars, aug_vars, alternative='less')

# Cohen's d (effect size cho paired comparison)
diff = np.array(aug_vars) - np.array(orig_vars)
cohens_d = diff.mean() / (diff.std(ddof=1) + 1e-8)

print(f"\nPaired t-test: t={t_stat:.3f}, p={t_p:.4f}")
print(f"Wilcoxon signed-rank: W={w_stat:.1f}, p={w_p:.4f}")
print(f"Cohen's d = {cohens_d:.3f} (effect size: {'LON' if abs(cohens_d)>0.8 else 'TRUNG BINH' if abs(cohens_d)>0.5 else 'NHO'})")
print(f"\n=> KET LUAN: Augmentation {'lam tang diversity CO Y NGHIA' if t_p < 0.05 else 'KHONG du bang chung tang diversity'}")

---
## 5. Tổng kết Preprocessing

In [ ]:
print("=" * 60)
print("TONG KET PREPROCESSING - ABLATION STUDY")
print("=" * 60)
print()
print("1. RESIZE:")
print(f"   224x224 la optimal (SSIM cao nhat)")
print(f"   ANOVA F={f_val:.2f}, Eta²={eta2_resize:.3f}, post-hoc: moi cap deu khac biet")
print()
print("2. COLOR SPACE:")
print(f"   So sanh PCA explained variance: xem bang components")
print()
print("3. NORMALIZATION:")
print(f"   Z-Score & ImageNet co KS stat thap nhat -> gan normal nhat")
print(f"   Levene test: variance khac nhau giua cac phuong phap")
print()
print("4. AUGMENTATION:")
print(f"   Pipeline 5 phep augmentation tang diversity")
print(f"   Paired t-test p={t_p:.4f}, Cohen's d={cohens_d:.3f}")
print("=" * 60)